# 08 — Real-World Eye Disease Prediction V2

This notebook demonstrates the final trained **EfficientNetV2-B0** model on **new, previously unseen retinal/fundus images**.

> **Important:** This model is trained for **retinal/fundus images** (fundus photography). Do not use general phone-camera eye or face images as input.

**Classes:** Normal | Cataract | Diabetic Retinopathy | Glaucoma

**Verified test accuracy:** 84.94% | **Glaucoma recall:** 57.73% | **Macro ROC-AUC:** 95.91%

> This notebook is for **inference demonstration only**. The model is not retrained or modified.

In [1]:
# STEP 1 - Imports and configuration
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r'D:/Practice Projects/Disease Detection')
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'model').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project root.')

MODEL_PATH = PROJECT_ROOT / 'model' / 'efficientnetv2_b0_4class_best.keras'
IMAGE_SIZE = (224, 224)
CLASS_NAMES = ['Normal', 'Cataract', 'Diabetic Retinopathy', 'Glaucoma']
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'real_world_prediction'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('MODEL_PATH:', MODEL_PATH)
print('IMAGE_SIZE:', IMAGE_SIZE)
print('CLASS_NAMES:', CLASS_NAMES)
print('REPORTS_DIR:', REPORTS_DIR)

TensorFlow: 2.21.0
PROJECT_ROOT: d:\Practice Projects\Disease Detection
MODEL_PATH: d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
IMAGE_SIZE: (224, 224)
CLASS_NAMES: ['Normal', 'Cataract', 'Diabetic Retinopathy', 'Glaucoma']
REPORTS_DIR: d:\Practice Projects\Disease Detection\reports\real_world_prediction


In [2]:
# STEP 2 - Load final model
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

model = keras.models.load_model(MODEL_PATH)

if int(model.output_shape[-1]) != 4:
    raise ValueError(f'Expected 4 output classes, got {model.output_shape[-1]}')

print(f'Model path   : {MODEL_PATH}')
print(f'Input shape  : {model.input_shape}')
print(f'Output shape : {model.output_shape}')
print(f'Class names  : {CLASS_NAMES}')
print('4-class model validation: PASSED')

Model path   : d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
Input shape  : (None, 224, 224, 3)
Output shape : (None, 4)
Class names  : ['Normal', 'Cataract', 'Diabetic Retinopathy', 'Glaucoma']
4-class model validation: PASSED


In [3]:
# STEP 3 - Robust image preprocessing function
# Pipeline: PIL open -> RGB -> bilinear resize 224x224 -> float32 -> batch dim
# No external /255: EfficientNetV2 model contains internal Rescaling layer.

def load_and_preprocess_image(image_path):
    """
    Load a retinal/fundus image and prepare it for model inference.
    Returns (original_rgb_image, model_input_tensor).
    """
    path = Path(str(image_path))
    if not path.exists():
        raise FileNotFoundError(f'Image not found: {path}')
    supported = {'.jpg', '.jpeg', '.png'}
    if path.suffix.lower() not in supported:
        raise ValueError(f'Unsupported extension: {path.suffix}. Use: {supported}')
    try:
        img = Image.open(path).convert('RGB')
    except Exception as e:
        raise ValueError(f'Could not open image {path}: {e}')
    img_resized = img.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
    arr = np.asarray(img_resized, dtype=np.float32)
    tensor = np.expand_dims(arr, axis=0)
    return img_resized, tensor

print('load_and_preprocess_image: READY')
print('Preprocessing: PIL BILINEAR resize -> float32 -> batch dim (no external /255)')

load_and_preprocess_image: READY
Preprocessing: PIL BILINEAR resize -> float32 -> batch dim (no external /255)


In [4]:
# STEP 4 - Prediction function

def predict_disease(image_path):
    """
    Run inference on a single retinal/fundus image.
    Returns dict: disease_name, class_id, confidence, all_probabilities.
    """
    _, tensor = load_and_preprocess_image(image_path)
    probs = model.predict(tensor, verbose=0)[0]
    class_id = int(np.argmax(probs))
    return {
        'disease_name': CLASS_NAMES[class_id],
        'class_id': class_id,
        'confidence': float(probs[class_id]),
        'all_probabilities': {CLASS_NAMES[i]: float(probs[i]) for i in range(4)}
    }

print('predict_disease: READY')

predict_disease: READY


In [5]:
# STEP 5 - Display prediction visualization

def display_prediction(image_path, result):
    img_rgb, _ = load_and_preprocess_image(image_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img_rgb)
    axes[0].set_title(
        f"Retinal/Fundus Image\nPredicted: {result['disease_name']}\nConfidence: {result['confidence']:.2%}",
        fontsize=12, fontweight='bold'
    )
    axes[0].axis('off')

    names = list(result['all_probabilities'].keys())
    probs = list(result['all_probabilities'].values())
    colors = ['#2ecc71' if n == result['disease_name'] else '#95a5a6' for n in names]
    bars = axes[1].barh(names, [p * 100 for p in probs], color=colors)
    axes[1].set_xlabel('Probability (%)')
    axes[1].set_title('Class Probabilities', fontsize=12)
    axes[1].set_xlim(0, 100)
    for bar, p in zip(bars, probs):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                     f'{p:.2%}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f"\nPredicted Disease : {result['disease_name']}")
    print(f"Confidence        : {result['confidence']:.2%}")
    print()
    for name, prob in result['all_probabilities'].items():
        print(f'  {name:<25}: {prob:.2%}')

print('display_prediction: READY')

display_prediction: READY


In [6]:
# STEP 6 - Confidence interpretation
# Note: confidence is a model probability score, not clinical certainty.

def interpret_confidence(confidence):
    if confidence >= 0.90:
        return 'High confidence prediction'
    elif confidence >= 0.70:
        return 'Moderate confidence prediction'
    else:
        return 'Low confidence prediction - review recommended'

print('interpret_confidence: READY')
print('  >= 90%  : High confidence prediction')
print('  70-89%  : Moderate confidence prediction')
print('  < 70%   : Low confidence prediction - review recommended')

interpret_confidence: READY
  >= 90%  : High confidence prediction
  70-89%  : Moderate confidence prediction
  < 70%   : Low confidence prediction - review recommended


In [7]:
# STEP 7 - Grad-CAM integration
# Target layer: top_activation (verified in 07_gradcam_v2.ipynb)
# Architecture is flat - top_activation is a direct child of the top-level model.

GRADCAM_LAYER = 'top_activation'
target_layer = model.get_layer(GRADCAM_LAYER)
disease_output_layer = model.get_layer('disease_output')

grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[target_layer.output, disease_output_layer.output]
)

def make_gradcam_heatmap(tensor, class_idx):
    with tf.GradientTape() as tape:
        feature_maps, predictions = grad_model(tensor, training=False)
        tape.watch(feature_maps)
        class_score = predictions[:, class_idx]
    grads = tape.gradient(class_score, feature_maps)
    if grads is None:
        raise ValueError('Gradients are None - Grad-CAM failed.')
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
    heatmap = tf.reduce_sum(
        feature_maps * pooled_grads[:, tf.newaxis, tf.newaxis, :], axis=-1
    )
    heatmap = tf.nn.relu(heatmap)[0].numpy()
    max_val = heatmap.max()
    if max_val > 1e-8:
        heatmap = heatmap / max_val
    return heatmap

def display_gradcam(image_path, result, save_path=None):
    img_rgb, tensor = load_and_preprocess_image(image_path)
    heatmap = make_gradcam_heatmap(tensor, result['class_id'])
    if heatmap.max() <= 1e-8:
        raise ValueError('Grad-CAM heatmap is effectively zero.')

    base = np.asarray(img_rgb, dtype=np.float32) / 255.0
    heatmap_img = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255))
    heatmap_img = heatmap_img.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
    heatmap_resized = np.asarray(heatmap_img, dtype=np.float32) / 255.0
    colored = plt.get_cmap('jet')(heatmap_resized)[..., :3]
    overlay = np.clip(0.60 * base + 0.40 * colored, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f"Grad-CAM - Predicted: {result['disease_name']} ({result['confidence']:.2%})",
        fontsize=13, fontweight='bold'
    )
    axes[0].imshow(base)
    axes[0].set_title('Original Retinal Image')
    axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Grad-CAM Heatmap\n(red = high activation)')
    axes[1].axis('off')
    axes[2].imshow(overlay)
    axes[2].set_title('Grad-CAM Overlay\n(retinal regions influencing prediction)')
    axes[2].axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Grad-CAM saved: {save_path}')
    plt.show()
    return heatmap

print(f'Grad-CAM target layer : {GRADCAM_LAYER}')
print(f'Target output shape   : {target_layer.output.shape}')
print('make_gradcam_heatmap  : READY')
print('display_gradcam       : READY')

Grad-CAM target layer : top_activation
Target output shape   : (None, 7, 7, 1280)
make_gradcam_heatmap  : READY
display_gradcam       : READY


In [8]:
# STEP 8 - REAL-WORLD SINGLE IMAGE DEMO
# Set sample_image_path to the path of your retinal/fundus image.
# Supported formats: .jpg, .jpeg, .png

sample_image_path = 'path/to/retinal_image.jpg'  # <-- replace with your image path

try:
    result = predict_disease(sample_image_path)

    print('=' * 50)
    print('REAL-WORLD SINGLE IMAGE DEMO')
    print('=' * 50)
    print(f"Image             : {sample_image_path}")
    print(f"Predicted Disease : {result['disease_name']}")
    print(f"Confidence        : {result['confidence']:.2%}")
    print(f"Interpretation    : {interpret_confidence(result['confidence'])}")
    print()
    for name, prob in result['all_probabilities'].items():
        print(f'  {name:<25}: {prob:.2%}')
    print('=' * 50)

    display_prediction(sample_image_path, result)
    display_gradcam(
        sample_image_path, result,
        save_path=REPORTS_DIR / 'demo_gradcam.png'
    )

except FileNotFoundError as e:
    print(f'[SKIPPED] Image not found: {e}')
    print('Set sample_image_path to a valid retinal/fundus image path to run the demo.')

[SKIPPED] Image not found: Image not found: path\to\retinal_image.jpg
Set sample_image_path to a valid retinal/fundus image path to run the demo.


In [9]:
# STEP 9 - Folder-based batch prediction

def predict_folder(folder_path, save_csv=True):
    """
    Run prediction on all .jpg/.jpeg/.png images in a folder.
    Saves results to reports/real_world_predictions.csv.
    """
    folder = Path(str(folder_path))
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder}')

    image_files = sorted(
        p for p in folder.iterdir()
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    )
    if not image_files:
        print(f'No supported images found in: {folder}')
        return pd.DataFrame()

    rows = []
    for img_path in image_files:
        try:
            result = predict_disease(img_path)
            probs = result['all_probabilities']
            rows.append({
                'image_name': img_path.name,
                'predicted_class_id': result['class_id'],
                'predicted_disease': result['disease_name'],
                'confidence': result['confidence'],
                'normal_probability': probs['Normal'],
                'cataract_probability': probs['Cataract'],
                'diabetic_retinopathy_probability': probs['Diabetic Retinopathy'],
                'glaucoma_probability': probs['Glaucoma'],
            })
            print(f"  {img_path.name:<40} -> {result['disease_name']} ({result['confidence']:.2%})")
        except Exception as e:
            print(f'  [ERROR] {img_path.name}: {e}')

    df = pd.DataFrame(rows)
    if save_csv and not df.empty:
        out_path = REPORTS_DIR / 'real_world_predictions.csv'
        df.to_csv(out_path, index=False)
        print(f'\nSaved: {out_path}')
    return df

# Usage: batch_df = predict_folder('path/to/retinal_images_folder')
print('predict_folder: READY')
print('Usage: batch_df = predict_folder("path/to/folder")')

predict_folder: READY
Usage: batch_df = predict_folder("path/to/folder")


In [10]:
# STEP 10 - Error handling validation

error_cases = [
    ('missing_file.jpg', 'Missing file'),
    ('image.bmp',        'Unsupported extension'),
]
for path, label in error_cases:
    try:
        predict_disease(path)
        print(f'  [{label}] No error raised (unexpected)')
    except (FileNotFoundError, ValueError) as e:
        print(f'  [{label}] Caught correctly: {type(e).__name__}: {e}')

print('\nError handling: PASSED')

  [Missing file] Caught correctly: FileNotFoundError: Image not found: missing_file.jpg
  [Unsupported extension] Caught correctly: FileNotFoundError: Image not found: image.bmp

Error handling: PASSED


In [11]:
# STEP 11 - Save prediction metadata

metadata = {
    'model_name': 'efficientnetv2_b0_4class_best',
    'model_path': str(MODEL_PATH),
    'image_size': list(IMAGE_SIZE),
    'class_mapping': {str(i): CLASS_NAMES[i] for i in range(4)},
    'preprocessing': (
        'PIL Image.open -> convert RGB -> resize 224x224 BILINEAR -> '
        'np.asarray float32 -> expand_dims batch. '
        'No external /255: model contains internal EfficientNetV2 Rescaling layer.'
    ),
    'gradcam_target_layer': 'top_activation',
    'final_verified_test_accuracy': 0.8494,
    'final_verified_glaucoma_recall': 0.5773,
    'final_verified_macro_roc_auc': 0.9591,
    'test_images': 883,
    'correct_predictions': 750,
    'incorrect_predictions': 133
}

meta_path = REPORTS_DIR / 'real_world_prediction_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata saved: {meta_path}')
for k, v in metadata.items():
    print(f'  {k}: {v}')

Metadata saved: d:\Practice Projects\Disease Detection\reports\real_world_prediction\real_world_prediction_metadata.json
  model_name: efficientnetv2_b0_4class_best
  model_path: d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
  image_size: [224, 224]
  class_mapping: {'0': 'Normal', '1': 'Cataract', '2': 'Diabetic Retinopathy', '3': 'Glaucoma'}
  preprocessing: PIL Image.open -> convert RGB -> resize 224x224 BILINEAR -> np.asarray float32 -> expand_dims batch. No external /255: model contains internal EfficientNetV2 Rescaling layer.
  gradcam_target_layer: top_activation
  final_verified_test_accuracy: 0.8494
  final_verified_glaucoma_recall: 0.5773
  final_verified_macro_roc_auc: 0.9591
  test_images: 883
  correct_predictions: 750
  incorrect_predictions: 133


In [12]:
# STEP 12 - Final validation

checks = {}

# 1. Model loaded
checks['model_loaded'] = model is not None

# 2-4. Prediction returns 4 finite probabilities summing to 1
try:
    _test_csv = PROJECT_ROOT / 'preprocessing' / 'splits_v2' / 'test.csv'
    _test_df = pd.read_csv(_test_csv)
    _sample_path = Path(str(_test_df.iloc[0]['image_path']))
    if not _sample_path.exists():
        _sample_path = PROJECT_ROOT / _sample_path
    _result = predict_disease(_sample_path)
    _probs = list(_result['all_probabilities'].values())
    checks['four_probabilities'] = len(_probs) == 4
    checks['probs_finite'] = all(float('inf') > p > float('-inf') for p in _probs)
    checks['probs_sum_to_one'] = abs(sum(_probs) - 1.0) < 1e-4
except Exception as e:
    checks['four_probabilities'] = False
    checks['probs_finite'] = False
    checks['probs_sum_to_one'] = False
    print(f'Prediction check error: {e}')

# 5. Class mapping correct
checks['class_mapping'] = CLASS_NAMES == ['Normal', 'Cataract', 'Diabetic Retinopathy', 'Glaucoma']

# 6. Grad-CAM target layer exists
try:
    model.get_layer('top_activation')
    checks['gradcam_layer_exists'] = True
except ValueError:
    checks['gradcam_layer_exists'] = False

# 7. Grad-CAM produces non-zero heatmap
try:
    _, _tensor = load_and_preprocess_image(_sample_path)
    _heatmap = make_gradcam_heatmap(_tensor, _result['class_id'])
    checks['gradcam_nonzero'] = bool(_heatmap.max() > 1e-8)
except Exception as e:
    checks['gradcam_nonzero'] = False
    print(f'Grad-CAM check error: {e}')

all_pass = all(checks.values())
pipeline_status = 'PASS' if all_pass else 'FAIL'

print('REAL-WORLD PREDICTION PIPELINE')
print('=' * 30)
print('Model        : efficientnetv2_b0_4class_best.keras')
print(f'Input size   : {IMAGE_SIZE}')
print(f'Classes      : {CLASS_NAMES}')
print('Preprocessing: PIL BILINEAR -> float32 -> batch (no external /255)')
print('Grad-CAM     : top_activation')
print('Test accuracy: 84.94%')
print('Glaucoma rec : 57.73%')
print(f'Pipeline     : {pipeline_status}')
print()
for check, passed in checks.items():
    print(f'  [{"PASS" if passed else "FAIL"}] {check}')

if not all_pass:
    raise AssertionError('One or more pipeline checks FAILED.')

print('\nREAL-WORLD PREDICTION NOTEBOOK COMPLETED SUCCESSFULLY')

d:\Practice Projects\Disease Detection\.venv311\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['image_input']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


REAL-WORLD PREDICTION PIPELINE
Model        : efficientnetv2_b0_4class_best.keras
Input size   : (224, 224)
Classes      : ['Normal', 'Cataract', 'Diabetic Retinopathy', 'Glaucoma']
Preprocessing: PIL BILINEAR -> float32 -> batch (no external /255)
Grad-CAM     : top_activation
Test accuracy: 84.94%
Glaucoma rec : 57.73%
Pipeline     : PASS

  [PASS] model_loaded
  [PASS] four_probabilities
  [PASS] probs_finite
  [PASS] probs_sum_to_one
  [PASS] class_mapping
  [PASS] gradcam_layer_exists
  [PASS] gradcam_nonzero

REAL-WORLD PREDICTION NOTEBOOK COMPLETED SUCCESSFULLY
